In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")

import yaml
import torch

In [ ]:
# Generación de datos sintéticos
from data_generation.data_config import DATA_CONFIG
from data_generation.simulator import DataSimulator

# Datos sintéticos -> preparados para entrenamiento
from splits.split_generator import SplitGenerator
from connectors.lstm import LSTMConnector
from datasets.panel_sequence import PanelSequenceDataset

# Entrenamiento y evaluación del modelo
from models.lstm_classifier import LSTMClassifier
from training.trainer import Trainer
from training.tuner import Tuner

In [ ]:
data_simulator = DataSimulator(DATA_CONFIG)
panel_data = data_simulator.simulate()

In [ ]:
split_generator = SplitGenerator(panel_data, train_nini_ratio=0.3, seed=13)
train_ids, test_ids = split_generator.generate()

In [ ]:
feature_cols = [
    "antiguedad",
    "ratio_formalidad",
    "empleados",
    "salario_promedio",
    "exportadora",
    "sector",
    "region"
]

connector = LSTMConnector(
    panel=panel_data,
    split=split_generator.split,
    feature_cols=feature_cols
)
train_dataset, test_dataset = connector.convert(fit_scaler=False)

In [ ]:
with open("../experiments/exp_000.yaml") as f:
    cfg = yaml.safe_load(f)

training_cfg = cfg["training"]
optuna_cfg = cfg["optuna"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

n_features = len(connector.feature_cols)
n_cohorts = len(connector._cohorts_periods)

tuner = Tuner(
    connector=connector,
    n_features=n_features,
    n_cohorts=n_cohorts,
    t_cfg=training_cfg,
    opt_cfg=optuna_cfg,
    device=device,
)

study = tuner.run(study_name="pipeline_test", storage="sqlite:///pipeline_test.db")

print(f"Best params: {study.best_trial.params}")
print(f"Best {optuna_cfg['metric']}: {study.best_trial.value:.4f}")
